In [4]:
import glob
import os
import pandas as pd # Assuming you're using this for the DataFrame on line 34
import numpy as np
import json
import requests
# Chemins vers les dossiers
path_normal = '../data/benign/'
path_attack = '../data/attack/'

def load_jsonl_files(directory, label):
    search_pattern = os.path.join(directory, "*.json")
    all_files = glob.glob(search_pattern)
    
    if not all_files:
        print(f"⚠️ Aucun fichier trouvé dans {directory}")
        return pd.DataFrame()

    data_list = []
    
    for filename in all_files:
        with open(filename, 'r', encoding='utf-8') as f:
            for line_number, line in enumerate(f, 1):
                line = line.strip()
                if not line: continue  # Ignore les lignes vides
                
                try:
                    # On décode chaque ligne individuellement
                    item = json.loads(line)
                    item['label'] = label
                    data_list.append(item)
                except json.JSONDecodeError:
                    # Parfois la première ou dernière ligne est mal formée
                    continue 

    print(f"✅ Chargé {len(data_list)} lignes depuis {directory}")
    return pd.DataFrame(data_list)

# 1. Chargement
df_normal = load_jsonl_files(path_normal, 0)
df_attack = load_jsonl_files(path_attack, 1)

# 2. Fusion
if not df_normal.empty or not df_attack.empty:
    df = pd.concat([df_normal, df_attack], ignore_index=True)
    print(f"\n🚀 Dataset finale prête ! Forme : {df.shape}")
else:
    print("❌ Erreur : Aucune donnée n'a pu être chargée.")

✅ Chargé 874923 lignes depuis ../data/benign/
✅ Chargé 290089 lignes depuis ../data/attack/

🚀 Dataset finale prête ! Forme : (1165012, 4)


In [5]:
# 3. Mélange et Sauvegarde
if not df.empty:
    # Mélanger les lignes (très important pour 2Go de données ordonnées)
    print("🎲 Mélange des données...")
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # Sauvegarde en CSV
    output_path = '../data/dataset_complet.csv'
    print(f"💾 Sauvegarde en cours dans {output_path}...")
    
    # index=False évite de créer une colonne inutile avec les numéros de ligne
    df.to_csv(output_path, index=False, encoding='utf-8')
    
    print(f"✅ Terminé ! Fichier sauvegardé ({os.path.getsize(output_path) / 1e9:.2f} Go)")

🎲 Mélange des données...
💾 Sauvegarde en cours dans ../data/dataset_complet.csv...
✅ Terminé ! Fichier sauvegardé (0.41 Go)
